# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashizhenya755-dev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
# SIGNAL 1 — Recent inactivity / staleness

staleness_data = con.sql(f"""
WITH page_history AS (
    SELECT
        content_hash_id,
        client_hash_id,
        MAX(report_date) AS last_observed_date
    FROM '{history_path}'
    WHERE report_date < DATE '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY content_hash_id, client_hash_id
),

march_pages AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM '{month_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
)

SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.march_impressions,
    m.march_clicks,
    DATE '2026-03-01' - p.last_observed_date AS days_since_last_activity
FROM march_pages m
JOIN page_history p
    ON m.content_hash_id = p.content_hash_id
   AND m.client_hash_id = p.client_hash_id
WHERE p.last_observed_date IS NOT NULL
""").df()

staleness_data["staleness_bucket"] = pd.cut(
    staleness_data["days_since_last_activity"],
    bins=[0, 2, 7, 30, float("inf")],
    labels=["1-2 days", "3-7 days", "8-30 days", "31+ days"],
    include_lowest=True
)

staleness_bucket_table = (
    staleness_data
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_march_impressions=("march_impressions", "mean"),
        median_march_impressions=("march_impressions", "median"),
        mean_march_clicks=("march_clicks", "mean"),
        median_march_clicks=("march_clicks", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS / RECENT INACTIVITY")
print(staleness_bucket_table.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — STALENESS / RECENT INACTIVITY
staleness_bucket      n  mean_march_impressions  median_march_impressions  mean_march_clicks  median_march_clicks
        1-2 days 110603             2352.483522                     474.0           6.682106                  1.0
        3-7 days  12961               50.097909                      13.0           0.170434                  0.0
       8-30 days  11106               37.227985                       5.0           0.130380                  0.0
        31+ days   6715               38.818168                       3.0           0.105733                  0.0


In [16]:
print("Verdict: CONFIRMED")
print(
    "Longer pre-March inactivity gaps are associated with substantially "
    "lower March search performance. This supports recent inactivity as "
    "a directional refresh-opportunity signal."
)

Verdict: CONFIRMED
Longer pre-March inactivity gaps are associated with substantially lower March search performance. This supports recent inactivity as a directional refresh-opportunity signal.


In [17]:
# SIGNAL 2 — Search position

position_audit = con.sql(f"""
WITH page_metrics AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position
    FROM '{month_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
)

SELECT *
FROM page_metrics
WHERE avg_position IS NOT NULL
  AND march_impressions > 0
""").df()

position_audit["position_bucket"] = pd.cut(
    position_audit["avg_position"],
    bins=[0, 5, 10, 20, float("inf")],
    labels=["1-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

position_bucket_table = (
    position_audit
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_march_impressions=("march_impressions", "mean"),
        median_march_impressions=("march_impressions", "median"),
        mean_march_clicks=("march_clicks", "mean"),
        median_march_clicks=("march_clicks", "median")
    )
    .reset_index()
)

print("SIGNAL 2 — SEARCH POSITION")
print(position_bucket_table.to_string(index=False))

SIGNAL 2 — SEARCH POSITION
position_bucket     n  mean_march_impressions  median_march_impressions  mean_march_clicks  median_march_clicks
            1-5 46572             2509.571481                     312.5           9.146590                  0.0
           5-10 55576             1307.341532                     158.0           3.882539                  0.0
          10-20 29922             1042.432291                     238.0           3.291491                  0.0
            20+ 44668             1341.751455                     117.0           1.826654                  0.0


In [18]:
print("Verdict: MIXED")
print(
    "Pages in better search positions generally show higher click volume, "
    "but impressions do not decrease consistently across all position "
    "buckets. Therefore, search position is supporting evidence rather "
    "than the primary baseline scoring signal."
)

Verdict: MIXED
Pages in better search positions generally show higher click volume, but impressions do not decrease consistently across all position buckets. Therefore, search position is supporting evidence rather than the primary baseline scoring signal.


### Signal audit summary

**Signal 1 — Recent inactivity: `CONFIRMED`**

Observed March search performance was substantially lower for pages with longer pre-March inactivity gaps. Median impressions fell from 474 for pages with a 1–2 day gap to 3 for pages with a 31+ day gap. This supports recent inactivity as a directional refresh-opportunity signal.

**Signal 2 — Search position: `MIXED`**

Search position showed a directional relationship with clicks: pages in positions 1–5 had higher mean clicks than pages at worse positions. However, impressions did not decrease consistently across all position buckets, so this signal is mixed rather than confirmed.

**Baseline decision**

Use recent inactivity as the primary scoring signal because it produced the clearest observed pattern. Treat search position as supporting evidence only and do not use it in the baseline score.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



**Plain-language rule**

Prioritize content pages that show a longer period of recent inactivity before March 2026. Pages with a larger pre-March activity gap receive a higher action score because the signal showed substantially lower March search impressions and clicks for these pages.

**Reason codes**

* `RECENT_INACTIVITY_HIGH` — 31+ days since the last observed activity before March.
* `RECENT_INACTIVITY_MEDIUM` — 8–30 days since the last observed activity.
* `RECENT_INACTIVITY_LOW` — 3–7 days since the last observed activity.
* `RECENT_INACTIVITY_NONE` — 1–2 days since the last observed activity.
* `NO_SIGNAL` — No usable recent-inactivity signal was available.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue = con.sql(f"""
WITH page_history AS (
    SELECT
        content_hash_id,
        client_hash_id,
        MAX(report_date) AS last_observed_date
    FROM '{history_path}'
    WHERE report_date < DATE '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY content_hash_id, client_hash_id
),

march_pages AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM '{month_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
),

scored AS (
    SELECT
        m.content_hash_id,
        m.client_hash_id,
        m.march_impressions,
        m.march_clicks,
        DATE '2026-03-01' - h.last_observed_date AS days_since_last_activity,

        CASE
            WHEN h.last_observed_date IS NULL THEN 0
            WHEN DATE '2026-03-01' - h.last_observed_date >= 31 THEN 3
            WHEN DATE '2026-03-01' - h.last_observed_date >= 8 THEN 2
            WHEN DATE '2026-03-01' - h.last_observed_date >= 3 THEN 1
            ELSE 0
        END AS score,

        CASE
            WHEN h.last_observed_date IS NULL THEN 'NO_SIGNAL'
            WHEN DATE '2026-03-01' - h.last_observed_date >= 31
                THEN 'RECENT_INACTIVITY_HIGH'
            WHEN DATE '2026-03-01' - h.last_observed_date >= 8
                THEN 'RECENT_INACTIVITY_MEDIUM'
            WHEN DATE '2026-03-01' - h.last_observed_date >= 3
                THEN 'RECENT_INACTIVITY_LOW'
            ELSE 'RECENT_INACTIVITY_NONE'
        END AS reason_code

    FROM march_pages m
    LEFT JOIN page_history h
        ON m.content_hash_id = h.content_hash_id
       AND m.client_hash_id = h.client_hash_id
)

SELECT
    content_hash_id,
    client_hash_id,
    march_impressions,
    march_clicks,
    days_since_last_activity,
    score,
    reason_code,

    CASE
        WHEN score >= 3 THEN 'REFRESH_HIGH_PRIORITY'
        WHEN score >= 2 THEN 'REFRESH'
        WHEN score >= 1 THEN 'REVIEW'
        ELSE 'MONITOR'
    END AS action,

    ROW_NUMBER() OVER (
        ORDER BY score DESC,
                 days_since_last_activity DESC NULLS LAST,
                 march_impressions DESC
    ) AS rank

FROM scored
ORDER BY rank
""").df()

# Save the ranked queue required by the assignment
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(ranked_queue))
print("Saved to:", output_path)
print()
print("Top 20:")
display(ranked_queue.head(20))



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue created successfully.
Rows: 176738
Saved to: work/outputs/baseline_action_score.csv

Top 20:


,content_hash_id,client_hash_id,march_impressions,march_clicks,days_since_last_activity,score,reason_code,action,rank
0,content_ad5dd150fc409643,client_ff644d8251367cbb,75.0,0.0,398,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,1
1,content_f345fc33637853ef,client_ff644d8251367cbb,22.0,0.0,398,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,2
2,content_636b7a200ed2fee8,client_ff644d8251367cbb,17.0,0.0,398,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,3
3,content_29fafc5744527fdf,client_ff644d8251367cbb,4.0,0.0,398,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,4
4,content_141ce15b96e45a24,client_ff644d8251367cbb,43.0,0.0,397,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,5
5,content_c4f735b5a733b956,client_ff644d8251367cbb,14.0,0.0,397,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,6
6,content_e38f4766e02da605,client_ff644d8251367cbb,12.0,0.0,397,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,7
7,content_13d3544ee207bf07,client_ff644d8251367cbb,8.0,0.0,397,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,8
8,content_53a6a3d4d126a10b,client_ff644d8251367cbb,7.0,0.0,397,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,9
9,content_2d702a5d92bc4677,client_ff644d8251367cbb,4.0,0.0,397,3,RECENT_INACTIVITY_HIGH,REFRESH_HIGH_PRIORITY,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-20 review
# Review the highest-ranked pages and record why they were selected
# and what could make the recommendation wrong.

top20_review = ranked_queue.head(20).copy()

top20_review["confidence_note"] = top20_review.apply(
    lambda row: (
        "Higher confidence: long inactivity gap with measurable March impressions."
        if row["march_impressions"] >= 20
        else
        "Lower confidence: long inactivity gap but very low March impressions."
    ),
    axis=1
)

top20_review["what_would_make_it_wrong"] = (
    "The inactivity gap may reflect missing/incomplete historical tracking "
    "rather than true content staleness; low March volume also makes the "
    "refresh opportunity uncertain."
)

review_columns = [
    "rank",
    "content_hash_id",
    "client_hash_id",
    "action",
    "reason_code",
    "score",
    "days_since_last_activity",
    "march_impressions",
    "march_clicks",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20_review[review_columns])


,rank,content_hash_id,client_hash_id,action,reason_code,score,days_since_last_activity,march_impressions,march_clicks,confidence_note,what_would_make_it_wrong
0,1,content_ad5dd150fc409643,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,398,75.0,0.0,Higher confidence: long inactivity gap with me...,The inactivity gap may reflect missing/incompl...
1,2,content_f345fc33637853ef,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,398,22.0,0.0,Higher confidence: long inactivity gap with me...,The inactivity gap may reflect missing/incompl...
2,3,content_636b7a200ed2fee8,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,398,17.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...
3,4,content_29fafc5744527fdf,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,398,4.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...
4,5,content_141ce15b96e45a24,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,397,43.0,0.0,Higher confidence: long inactivity gap with me...,The inactivity gap may reflect missing/incompl...
5,6,content_c4f735b5a733b956,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,397,14.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...
6,7,content_e38f4766e02da605,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,397,12.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...
7,8,content_13d3544ee207bf07,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,397,8.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...
8,9,content_53a6a3d4d126a10b,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,397,7.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...
9,10,content_2d702a5d92bc4677,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,3,397,4.0,0.0,Lower confidence: long inactivity gap but very...,The inactivity gap may reflect missing/incompl...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

print("WEAK PICKS")
print("-" * 60)

weak_picks = ranked_queue[
    (ranked_queue["score"] >= 3) &
    (ranked_queue["march_impressions"] < 20)
].head(10)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "client_hash_id",
            "action",
            "reason_code",
            "days_since_last_activity",
            "march_impressions",
            "march_clicks"
        ]
    ]
)

print()
print("Why these may be weak:")
print(
    "Some high-priority picks have very long inactivity gaps but very low "
    "March impressions. They may represent genuinely inactive content, "
    "but the low search volume makes the practical refresh opportunity uncertain."
)

print()
print("LEAKAGE CHECK")
print("-" * 60)

# Features used by the baseline
allowed_features = {
    "days_since_last_activity",
    "march_impressions",
    "march_clicks"
}

print("Baseline input features:", sorted(allowed_features))

# Confirm that future/label-derived fields were not used.
forbidden_terms = [
    "future",
    "label",
    "decline",
    "zero_click",
    "product_flag"
]

queue_text = " ".join(ranked_queue.columns).lower()

leakage_terms_found = [
    term for term in forbidden_terms
    if term in queue_text
]

if leakage_terms_found:
    print("WARNING — possible forbidden terms found:", leakage_terms_found)
else:
    print("PASS — no future-window, label-derived, or product-flag columns "
          "are present in the ranked queue.")

print()
print("No client names, URLs, or private queries were included.")


WEAK PICKS
------------------------------------------------------------


,rank,content_hash_id,client_hash_id,action,reason_code,days_since_last_activity,march_impressions,march_clicks
2,3,content_636b7a200ed2fee8,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,398,17.0,0.0
3,4,content_29fafc5744527fdf,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,398,4.0,0.0
5,6,content_c4f735b5a733b956,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,397,14.0,0.0
6,7,content_e38f4766e02da605,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,397,12.0,0.0
7,8,content_13d3544ee207bf07,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,397,8.0,0.0
8,9,content_53a6a3d4d126a10b,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,397,7.0,0.0
9,10,content_2d702a5d92bc4677,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,397,4.0,0.0
10,11,content_4c2928953c272913,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,397,1.0,0.0
14,15,content_1b5d6f74f16c6e77,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,396,10.0,0.0
15,16,content_7e6417b9b2586b8d,client_ff644d8251367cbb,REFRESH_HIGH_PRIORITY,RECENT_INACTIVITY_HIGH,396,3.0,0.0



Why these may be weak:
Some high-priority picks have very long inactivity gaps but very low March impressions. They may represent genuinely inactive content, but the low search volume makes the practical refresh opportunity uncertain.

LEAKAGE CHECK
------------------------------------------------------------
Baseline input features: ['days_since_last_activity', 'march_clicks', 'march_impressions']
PASS — no future-window, label-derived, or product-flag columns are present in the ranked queue.

No client names, URLs, or private queries were included.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.